# Interactive Visual Lab 2: The SwiGLU Gated Feed-Forward Network

Welcome! In this interactive notebook, we explore the **SwiGLU (Swish Gated Linear Unit)** architecture used in **Llama 3, DeepSeek, and Mirendil-Base** using **interactive plots, dimension tracers, and runnable PyTorch code**.

---

## 1. Setup & Environment

In [1]:
import sys
from pathlib import Path

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from IPython.display import display, Markdown

from src.model.mlp import SwiGLUFFN

plt.rcParams["figure.dpi"] = 110
print("PyTorch and interactive plotting ready!")

PyTorch and interactive plotting ready!


---
## 2. Interactive Activation Curves: ReLU vs GELU vs SiLU (Swish)

The gate in SwiGLU uses $\text{SiLU}(z) = z \cdot \sigma(z)$. Let's visualize how the gate values behave compared to older functions:

> **Try moving the slider below:**
> Watch the live point on the curves to see how negative and positive signals are passed through.

In [2]:
def plot_activations(test_input=1.0):
    z = np.linspace(-4, 4, 200)
    relu = np.maximum(0, z)
    gelu = 0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z**3)))
    silu = z * (1 / (1 + np.exp(-z)))

    val_relu = max(0, test_input)
    val_gelu = (
        0.5
        * test_input
        * (1 + np.tanh(np.sqrt(2 / np.pi) * (test_input + 0.044715 * test_input**3)))
    )
    val_silu = test_input * (1 / (1 + np.exp(-test_input)))

    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.plot(z, relu, label="ReLU: max(0, z)", color="#f85149", lw=2, linestyle="--")
    ax.plot(z, gelu, label="GELU (GPT-2)", color="#d29922", lw=2)
    ax.plot(z, silu, label="SiLU / Swish (Llama 3 / SwiGLU)", color="#58a6ff", lw=2.5)

    # Current point
    ax.plot(
        test_input,
        val_silu,
        "o",
        color="#3fb950",
        markersize=9,
        label=f"SiLU({test_input:.1f}) = {val_silu:.2f}",
    )
    ax.axvline(test_input, color="#8b949e", linestyle=":", alpha=0.5)
    ax.axhline(0, color="#8b949e", lw=1, alpha=0.4)

    ax.set_title(
        "Activation Functions: Notice the Smooth Negative Valley in SiLU",
        fontsize=12,
        fontweight="bold",
        pad=12,
    )
    ax.set_xlabel("Input Gate Signal (z)", fontsize=10)
    ax.set_ylabel("Gate Multiplier Output", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left")

    plt.tight_layout()
    plt.show()

    display(Markdown(f"""
> ### Current Value at $z = {test_input:.2f}$:
> - **ReLU:** `{val_relu:.4f}` *(Hard cutoff to 0 if negative)*
> - **GELU:** `{val_gelu:.4f}`
> - **SiLU:** `{val_silu:.4f}` *(Allows a smooth, negative gradient around -1.0 to prevent dying neurons)*
"""))


interact(
    plot_activations,
    test_input=FloatSlider(
        min=-3.0, max=3.0, step=0.2, value=1.0, description="Gate Input (z):"
    ),
)

interactive(children=(FloatSlider(value=1.0, description='Gate Input (z):', max=3.0, min=-3.0, step=0.2), Outp…

<function __main__.plot_activations(test_input=1.0)>

---
## 3. The SwiGLU Architecture Walkthrough

The complete equation is:
$$\text{SwiGLU}(x) = \Big( \text{SiLU}(x W_{\text{gate}}) \odot (x W_{\text{up}}) \Big) W_{\text{down}}$$

Let's pass a real tensor through our `SwiGLUFFN` layer and trace each dimension:

In [3]:
dim = 768  # Model hidden dimension

# 8/3 Scaling rule rounded up to multiple of 256
raw_hidden = int(2 * (4 * dim) / 3)
aligned_hidden = 256 * ((raw_hidden + 256 - 1) // 256)

print(f"Model Dimension:       {dim}")
print(f"Raw 8/3 Hidden Dim:    {raw_hidden}")
print(f"GPU-Aligned Hidden Dim: {aligned_hidden} (multiple of 256)")

# Instantiate layer
ffn = SwiGLUFFN(dim=dim, hidden_dim=aligned_hidden)

# Input batch: 2 sequences, 4 tokens each
x = torch.randn(2, 4, dim)
print(f"\nInput Shape x:                  {list(x.shape)}")

# Step-by-step trace
gate_linear = ffn.w_gate(x)
gate_silu = F.silu(gate_linear)
up_features = ffn.w_up(x)
gated_features = gate_silu * up_features
final_output = ffn.w_down(gated_features)

print(f"1. Gate Projection w_gate(x):   {list(gate_linear.shape)}")
print(f"2. Gate Activated SiLU(gate):   {list(gate_silu.shape)}")
print(f"3. Up Projection w_up(x):       {list(up_features.shape)}")
print(f"4. Gated Product (2 * 3):       {list(gated_features.shape)}")
print(f"5. Final Projection w_down:     {list(final_output.shape)}")

# Verify full forward call matches trace
assert torch.allclose(final_output, ffn(x))
print("\nSUCCESS: Output verified to match layer forward pass!")

Model Dimension:       768
Raw 8/3 Hidden Dim:    2048
GPU-Aligned Hidden Dim: 2048 (multiple of 256)

Input Shape x:                  [2, 4, 768]
1. Gate Projection w_gate(x):   [2, 4, 2048]
2. Gate Activated SiLU(gate):   [2, 4, 2048]
3. Up Projection w_up(x):       [2, 4, 2048]
4. Gated Product (2 * 3):       [2, 4, 2048]
5. Final Projection w_down:     [2, 4, 768]

SUCCESS: Output verified to match layer forward pass!
